In [134]:
%load_ext autoreload
%autoreload 2
import jax.numpy as jnp
import jax_sn
import jax_sbgeom as jsb
import jax
import matplotlib.pyplot as plt
import numpy as onp
import equinox as eqx
jax.config.update("jax_enable_x64", True)
from jax_sbgeom.flux_surfaces import FluxSurfaceNormalExtendedNoPhi, ToroidalExtent
vmec_file = ["/home/tbogaarts/stellarator_paper/base_data/vmecs/helias5_vmec.nc4"][0]
fs_base = FluxSurfaceNormalExtendedNoPhi.from_hdf5(vmec_file)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [135]:
n_plasma = 10
n_sol = 2
s_mesh      = jnp.concatenate([jnp.linspace(0.0, 1.0, 10)**2, jnp.linspace(1.0, 1.2, n_sol+1)[1:]])
tor_extent  = ToroidalExtent.half_module(fs_base)
n_theta     = 51
n_phi       = 62
source_kwargs = {
    "n0"    : 2.4e20,
    "Ti0"   : 15.2,
    "nalpha": 0.35,
    "Tialpha": 1.2,
}

mesh_plasma = jsb.flux_surfaces.mesh_tetrahedra(fs_base, s_mesh, tor_extent, n_theta, n_phi)

In [136]:
s_values, theta_values, phi_values = jsb.flux_surfaces.flux_surface_meshing._mesh_tetrahedra_points_coordinates(fs_base, s_mesh, True, *tor_extent, n_theta, n_phi)
fs_values = jnp.nan_to_num(jsb.interfaces.plasma.flux_surface_reaction_rates_simple(s_values, **source_kwargs), nan = 0.0)
fs_values_elem = jnp.mean(fs_values[mesh_plasma[1]], axis=-1)

In [137]:
from jax_sbgeom.jax_utils import mesh_to_pyvista_mesh
import pyvista as pv
plotter  = pv.Plotter(shape = (1,2))
plotter.subplot(0,0)
plotter.add_mesh(mesh_to_pyvista_mesh(mesh_plasma), scalars=fs_values, cmap="viridis", show_scalar_bar=True)
plotter.subplot(0,1)
plotter.add_mesh(mesh_to_pyvista_mesh(mesh_plasma), scalars=fs_values_elem, cmap="viridis", show_scalar_bar=True)
#plotter.show()
plotter.close()

In [138]:
from jax_sn.solution_domain import BasixLagrangianSimplex
from jax_sn.domain import GeometricDomain, CrossSectionsIndexed, CrossSectionData
element                   = jax_sn.solution_domain.BasixLagrangianSimplex(1,3)
geometric_domain          = jax_sn.domain.GeometricDomain.from_mesh(mesh_plasma, element.face_template)
batched_xs                = jax.tree_util.tree_map(lambda *x : jnp.stack(x), CrossSectionData.zero(1,0))
cross_sections_idx        = CrossSectionsIndexed(jnp.arange(0, geometric_domain.n_elements), batched_xs)
domain                    = jax_sn.domain.Domain.from_geometric_domain_and_cross_sections(geometric_domain, cross_sections_idx)
quadrature_set            = jax_sn.quadrature_set.create_tn_quadrature_set(3)
topological_sorted_angles = jax_sn.domain.TopologicalSortedAngle.from_domain_quadrature_set(domain, quadrature_set)
solution_domain           = jax_sn.solution_domain.SolutionDomain.from_element_and_domain(element, domain)
source_eg0                = jnp.repeat(fs_values_elem, repeats=solution_domain.n_basis, axis=-1).reshape(domain.n_elements, solution_domain.n_basis)
source_eg0_all_angles     = jnp.stack([source_eg0]*quadrature_set.n_angles, axis=0)

In [ ]:
from jax_sbgeom.flux_surfaces import FluxSurface


def _normalise_theta_phi(theta : jnp.ndarray, phi : jnp.ndarray, toroidal_extent):
    return (theta % (2 * jnp.pi)) / (2 * jnp.pi), ((phi - toroidal_extent.start) % (toroidal_extent.end - toroidal_extent.start)) / (toroidal_extent.end - toroidal_extent.start)

@eqx.filter_jit
def create_sol_theta(d_grid : jnp.ndarray, flux_surface : FluxSurface, n_theta : int, n_phi : int, n_sol : int, toroidal_extent : ToroidalExtent):        
    theta_mg, phi_mg             = jnp.meshgrid(jnp.linspace(0, 2 * jnp.pi, n_theta, endpoint=False), jnp.linspace(toroidal_extent.start, toroidal_extent.end, n_phi, endpoint=True), indexing='ij')
    
    theta_sol_norm, phi_sol_norm = _normalise_theta_phi(theta_mg, phi_mg, toroidal_extent)

    d_edge                       =  jsb.flux_surfaces.flux_surfaces_base.bilinear_interp(theta_sol_norm, phi_sol_norm, d_grid)
    s_linspace                   = jnp.linspace(0, 1, n_sol +1)[1:]
    locations_sol                = flux_surface.cartesian_position(1.0 + d_edge[None, :, :] * s_linspace[:, None, None], theta_mg[ :, :], phi_mg[:, :])
    return locations_sol


d_grid = jnp.ones((10,10)) * 0.2
sol_vertices = create_sol_theta(d_grid, fs_base, n_theta, n_phi,  n_sol,  tor_extent )    

In [151]:
n_soltotal = n_sol * n_phi * n_theta
onp.testing.assert_allclose(mesh_plasma[0][-n_soltotal:, :], sol_vertices.reshape(-1,3), rtol=1e-5, atol=1e-8)

In [69]:
from jax_sn.operators.transport_operator import TransportOperatorMultipleAnglesVmap, TransportOperatorBatchedPallas
from jax_sn.operators.moment_operator import MomentOperator
batch_size         = 100
transport_operator = TransportOperatorMultipleAnglesVmap(TransportOperatorBatchedPallas(solution_domain.solution_domain_data, topological_sorted_angles,0, batch_size))
mass_operator      = jax_sn.operators.mass_operator.MassOperatorVmap.from_transport_operator(transport_operator)
moment_operator    = MomentOperator.from_quadrature_set(quadrature_set, l_max=0, elements_basis_shape= solution_domain.elements_basis_shape)

In [46]:
HQ = mass_operator.mv(source_eg0_all_angles)

In [71]:
fluxes = moment_operator.mv(transport_operator.inv(mass_operator.mv(source_eg0_all_angles)))

In [72]:
fluxes_0_points = solution_domain.interpolate(mesh_plasma[0], fluxes[0])

In [73]:
plotter = pv.Plotter()
plotter.add_mesh(mesh_to_pyvista_mesh(mesh_plasma), scalars=fluxes_0_points, cmap="viridis", show_scalar_bar=True, clim = [3.54e17, 8e18])
plotter.show()
#plotter.close()

Widget(value='<iframe src="http://localhost:46345/index.html?ui=P_0x7ea46a1647d0_8&reconnect=auto" class="pyvi…

In [ ]:
import StellBlanket as SB
import StellBlanket.SBGeom as SBGeom
import stellsim
def create_SB_version(vmec_file, tn_order, n_plasma, n_sol, fw_distance, n_theta, n_phi, s_power_sampling, toroidal_extent , solution_order, L_order, source_kwargs):
    flux_surface = SBGeom.VMEC.Flux_Surfaces_From_HDF5(vmec_file)
    flux_surface = SBGeom.Flux_Surfaces_Normal_Extended_No_Phi(flux_surface)
    energy_set = SB.Generic_Elements.Energy_Set([1,0])
    quadrature_set = SB.Generic_Elements.TN_Quadrature_Set(tn_order)
    materials = SB.Generic_Elements.Material(SB.Generic_Elements.Vacuum_Cross_Section())
    
    s_mesh = onp.concatenate([onp.linspace(0.0, 1.0, n_plasma)**s_power_sampling, onp.linspace(1.0, 1.0 + fw_distance, n_sol+1)[1:]])

    domain = SB.Tetrahedrons.Flux_Surface_Tetrahedron_Domain(
        flux_surface,
        n_phi=n_phi,
        n_theta=n_theta,
        phi_min=toroidal_extent[0],
        phi_max=toroidal_extent[1],
        s_layers=onp.where(s_mesh > 1.0, 1.0, s_mesh),
        lcfs_distance_label_Layers=onp.where(s_mesh > 1.0, s_mesh - 1.0, 0.0),
        material_Layers=[materials for _ in range(s_mesh.shape[0] - 1)]
    )
    matrixfreente = SB.Generic_Elements.Create_RAM_Simulation(quadrature_set, energy_set, domain, solution_order, L_order, boundary_condition=None)
    source = SB.Tetrahedrons.Construct_Flux_Surface_Source_14MeV_Simple(domain, **source_kwargs)

    return matrixfreente, source

matfreente, source = create_SB_version(vmec_file, 3, n_plasma, n_sol, 0.2,  n_theta, n_phi, 2, (tor_extent.start, tor_extent.end), 1,0, source_kwargs)
onp.testing.assert_allclose(matfreente.geometric_domain.vertices, mesh_plasma[0], rtol=1e-5, atol=1e-8)
onp.testing.assert_allclose(matfreente.geometric_domain.connectivity, mesh_plasma[1], rtol=1e-5, atol=1e-8)
SB.Generic_Elements.Run_Forward(matfreente, source)
tot_flux = stellsim.stellblanket_stell.average_fluxes(matfreente.global_moment_solution)
print(tot_flux.shape)
plotter = pv.Plotter()
plotter.add_mesh(mesh_to_pyvista_mesh(mesh_plasma), scalars=tot_flux[:,0] *onp.sqrt(4 * onp.pi), cmap="viridis", show_scalar_bar=True, clim = [3.54e17, 8e18])
plotter.show()